In [13]:
#To check the shape columns category distributions, fraud counts and missing values
import pandas as pd

df = pd.read_csv('data/paysim.csv')
print(df.shape)
print(df.columns.tolist())
print(df['type'].value_counts())
print(df['isFraud'].value_counts())
print(df.isnull().sum())

(6362620, 11)
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']
type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64
isFraud
0    6354407
1       8213
Name: count, dtype: int64
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64


In [17]:
df_relevant = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()

df_relevant['type_encoded'] = df_relevant['type'].map({'TRANSFER': 0, 'CASH_OUT': 1})
df_relevant['balance_diff_orig'] = df_relevant['oldbalanceOrg'] - df_relevant['amount'] - df_relevant['newbalanceOrig']
df_relevant['balance_diff_dest'] = df_relevant['oldbalanceDest'] + df_relevant['amount'] - df_relevant['newbalanceDest']

fraud = df_relevant[df_relevant['isFraud'] == 1]
legit = df_relevant[df_relevant['isFraud'] == 0].sample(n=50000, random_state=42)
sample = pd.concat([fraud, legit]).sample(frac=1, random_state=42)  # shuffle

features = [
    'type_encoded', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest', 'balance_diff_orig', 'balance_diff_dest'
]

X = sample[features]
y = sample['isFraud']

# Add these print statements so you can see your output!
print("X shape:", X.shape)
print("y value counts:")
print(y.value_counts())

X shape: (58213, 8)
y value counts:
isFraud
0    50000
1     8213
Name: count, dtype: int64


In [20]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight='balanced'   # crucial for handling class imbalance
)
model.fit(X_train, y_train)

ImportError: DLL load failed while importing _cyutility: An Application Control policy has blocked this file.